<a href="https://colab.research.google.com/github/memo124/Reto-de-Visi-n-Clasificaci-n-Automatizada-con-CNN/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reto de Visión: Clasificación Automatizada con CNN - E-commerce de Moda
## Parte 1: Preparación, Data Wrangling y Entrenamiento (Integrante 1)

In [ ]:
# 1. Importación de librerías necesarias
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt

print("Versión de TensorFlow:", tf.__version__)

### Carga de Datos y Definición de Clases
Importamos el dataset Fashion MNIST de forma nativa desde Keras y definimos las etiquetas comerciales para cada categoría del e-commerce.

In [ ]:
# Importación nativa de Fashion MNIST
fashion_mnist = keras.datasets.fashion_mnist
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# Definimos los nombres de las clases comerciales para uso posterior (Fase de evaluación)
class_names = ['Camiseta/Top', 'Pantalón', 'Suéter', 'Vestido', 'Abrigo',
               'Sandalia', 'Camisa', 'Zapatilla', 'Bolso', 'Botín']

print(f"Forma original de los datos de entrenamiento (Lote, Alto, Ancho): {X_train.shape}")

## Fase 1: Data Wrangling para Imágenes
Para que la red neuronal procese la información eficientemente y sin errores, debemos realizar dos pasos críticos:
1. **Normalización:** Escalar los valores de los píxeles al rango [0, 1].
2. **Reshape:** Ajustar los tensores a 4 dimensiones agregando el canal de color (escala de grises = 1).

In [ ]:
# 1. Normalización (Escalado de píxeles)
# Los píxeles originales van de 0 a 255. Dividimos entre 255.0 (float) para evitar pérdida de datos.
# Esto acelera la convergencia del modelo durante el entrenamiento.
X_train_norm = X_train / 255.0
X_test_norm = X_test / 255.0

# 2. Reshape (Redimensionamiento)
# Las capas Conv2D esperan tensores de forma: (Lote, Alto, Ancho, Canales).
# Añadimos la dimensión final '1' que representa el único canal de color (blanco y negro).
X_train_reshaped = X_train_norm.reshape((X_train_norm.shape[0], 28, 28, 1))
X_test_reshaped = X_test_norm.reshape((X_test_norm.shape[0], 28, 28, 1))

print(f" Forma de tensores de entrenamiento listos para la CNN: {X_train_reshaped.shape}")
print(f" Forma de tensores de prueba listos para la CNN: {X_test_reshaped.shape}")

## Fase 2: Diseño de la CNN y Entrenamiento
Construimos la arquitectura de la red neuronal intercalando capas convolucionales (extracción de características) con capas de agrupación (reducción de dimensionalidad). Finalizamos con capas densas para la clasificación final en 10 categorías.

In [ ]:
# Construcción del modelo Secuencial
model = models.Sequential()

# -- Bloque Extractor de Características --
# Capa Conv2D 1: 32 filtros, kernel 3x3, activación ReLU
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)))
# Capa MaxPooling2D 1: Reduce espacialmente a la mitad (2x2)
model.add(layers.MaxPooling2D((2, 2)))

# Capa Conv2D 2: 64 filtros para capturar patrones más complejos de la ropa
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
# Capa MaxPooling2D 2
model.add(layers.MaxPooling2D((2, 2)))

# -- Bloque Clasificador --
# Flatten: Aplana los mapas de características 2D a un vector 1D
model.add(layers.Flatten())

# Capa Densa (Oculta): Capa tradicional completamente conectada
model.add(layers.Dense(64, activation='relu'))

# Capa Densa (Salida): 10 neuronas (una por clase) y activación Softmax para obtener probabilidades
model.add(layers.Dense(10, activation='softmax'))

# Verificamos la estructura requerida
model.summary()

### Compilación y Entrenamiento del Modelo
Utilizamos el optimizador `adam` y la función de pérdida `sparse_categorical_crossentropy` (ya que nuestras etiquetas son números enteros del 0 al 9, no vectores one-hot). Entrenaremos por 10 épocas.

In [ ]:
# Compilación del modelo (requisitos de la rúbrica)
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print(" Iniciando el entrenamiento del modelo de Visión Computacional...")

# Entrenamiento (Fit)
# Asignamos el resultado a la variable 'history' para que el Integrante 2 pueda graficar las métricas.
# Pasamos los datos de prueba en validation_data para monitorear el sobreajuste época tras época.
history = model.fit(X_train_reshaped,
                    y_train,
                    epochs=10,
                    validation_data=(X_test_reshaped, y_test),
                    batch_size=64)

print("\n ¡Entrenamiento finalizado exitosamente!")
# Guardamos el modelo por precaución para facilitar el paso a paso en GitHub/Colab
model.save('fashion_mnist_cnn.keras')
print("Modelo guardado como 'fashion_mnist_cnn.keras'. Listo para la Fase 3 por el Integrante 2.")